# LeetCode 2150 — Find All Lonely Numbers in the Array bằng Python

Link đề: https://leetcode.com/problems/find-all-lonely-numbers-in-the-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def findLonely(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Dịch định nghĩa "số cô đơn" thành **ba điều kiện** kiểm tra được: xuất hiện đúng 1 lần, không có `x − 1`, không có `x + 1`.
2. Dùng **bảng đếm** (`Counter`, `dict`, mảng đếm) để trả lời "xuất hiện bao nhiêu lần?" trong O(1).
3. So sánh 4–5 cách: vét cạn O(n²), **sắp xếp + nhìn hàng xóm**, `Counter`, mảng đếm theo giá trị, `set` + `Counter`.
4. Xử lý cẩn thận **biên mảng** khi sắp xếp (phần tử đầu/cuối) và **biên giá trị** (`x = 0` ⇒ `x − 1 = −1`).
5. Test khi đáp án "theo thứ tự bất kỳ": so sánh **sau khi sắp xếp** (hoặc so multiset).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Số `x` là **cô đơn** nếu nó xuất hiện **đúng một lần** và **không có** số kề nó (`x + 1` và `x − 1`) xuất hiện trong mảng.

Trả về **mọi** số cô đơn (thứ tự tùy ý).

**Ví dụ 1:** `nums = [10,6,5,8]` → `[10,8]` (5 và 6 kề nhau nên không cô đơn).

**Ví dụ 2:** `nums = [1,3,5,3]` → `[1,5]` (3 xuất hiện hai lần).

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁶`.

In [1]:
from typing import List
from collections import Counter
import random, time

examples = [([10, 6, 5, 8], [10, 8]), ([1, 3, 5, 3], [1, 5])]
def same(a, b):
    return sorted(a) == sorted(b)

for nums, exp in examples:
    print(nums, "-> kỳ vọng (thứ tự tùy ý)", exp)

[10, 6, 5, 8] -> kỳ vọng (thứ tự tùy ý) [10, 8]
[1, 3, 5, 3] -> kỳ vọng (thứ tự tùy ý) [1, 5]


## 3. Kiến thức nền

### 3.1 `Counter` — đếm tần suất, truy cập khóa thiếu trả 0

In [2]:
cnt = Counter([1, 3, 5, 3])
print(cnt, cnt[3], cnt[2], "2" in cnt, 2 in cnt)

Counter({3: 2, 1: 1, 5: 1}) 2 0 False False


`cnt[2]` trả `0` **mà không thêm khóa** 2 vào `cnt` — an toàn khi đang duyệt `cnt.items()` (khác `defaultdict`).

### 3.2 `set` — kiểm tra tồn tại O(1)

In [3]:
s = set([10, 6, 5, 8])
print(9 in s, 5 in s, len(s))

False True 4


### 3.3 Sắp xếp và "nhìn hàng xóm"

Sau khi sắp xếp, các giá trị bằng nhau nằm cạnh nhau, và giá trị `x ± 1` (nếu có) cũng nằm **ngay cạnh** khối của `x`. Nên chỉ cần so `a[i]` với `a[i − 1]` và `a[i + 1]`.

In [4]:
a = sorted([10, 6, 5, 8, 6])
print(a)
for i, x in enumerate(a):
    left = a[i - 1] if i > 0 else None; right = a[i + 1] if i + 1 < len(a) else None
    print(f"a[{i}]={x}: trái={left}, phải={right}")

[5, 6, 6, 8, 10]
a[0]=5: trái=None, phải=6
a[1]=6: trái=5, phải=6
a[2]=6: trái=6, phải=8
a[3]=8: trái=6, phải=10
a[4]=10: trái=8, phải=None


### 3.4 Mảng đếm theo giá trị

Vì `0 ≤ nums[i] ≤ 10⁶`, có thể dùng list độ dài `10⁶ + 2` làm bảng đếm (tránh băm). Thêm 1 ô đệm để `x + 1` không vượt chỉ số; `x − 1 = −1` thì phải kiểm tra riêng (Python `a[-1]` là phần tử **cuối**, không lỗi nhưng **sai nghĩa**!).

In [5]:
cntarr = [0] * 8
for x in [0, 3, 3, 5]:
    cntarr[x] += 1
print(cntarr, "| cntarr[-1] =", cntarr[-1], "(là ô cuối, không phải 'số -1')")

[1, 0, 0, 2, 0, 1, 0, 0] | cntarr[-1] = 0 (là ô cuối, không phải 'số -1')


### 3.5 `itertools.groupby` — nhóm các giá trị bằng nhau liên tiếp

Sau khi sắp xếp, `groupby` cho từng khối giá trị bằng nhau cùng độ dài khối — một cách khác để biết "xuất hiện bao nhiêu lần" mà không cần bảng băm.

In [6]:
from itertools import groupby
a = sorted([1, 3, 5, 3, 7, 8, 10])
blocks = [(k, len(list(g))) for k, g in groupby(a)]
print(blocks)

[(1, 1), (3, 2), (5, 1), (7, 1), (8, 1), (10, 1)]


## 4. Ý tưởng nền tảng / trực giác

Kiểm tra từng số theo ba điều kiện. Làm tay ví dụ 1 `[10, 6, 5, 8]`:

| x | số lần | có x−1? | có x+1? | cô đơn? |
|---|---|---|---|---|
| 10 | 1 | 9: không | 11: không | ✔ |
| 6 | 1 | 5: **có** | 7: không | ✘ |
| 5 | 1 | 4: không | 6: **có** | ✘ |
| 8 | 1 | 7: không | 9: không | ✔ |

Ví dụ 2 `[1, 3, 5, 3]`: 3 xuất hiện 2 lần ⇒ ✘; 1: không có 0, 2 ⇒ ✔; 5: không có 4, 6 ⇒ ✔.

Cả ba điều kiện đều là câu hỏi "giá trị `v` xuất hiện bao nhiêu lần?" ⇒ một **bảng đếm** trả lời tất cả.

Lưu ý: số `3` lặp lại tuy **không** cô đơn nhưng vẫn **làm hàng xóm** cho số 2 và 4 (nếu có) mất cô đơn.

In [7]:
for nums, _ in examples:
    cnt = Counter(nums)
    for x in nums:
        print(f"x={x}: lần={cnt[x]}, có {x-1}? {cnt[x-1] > 0}, có {x+1}? {cnt[x+1] > 0} -> "
              f"{'cô đơn' if cnt[x] == 1 and not cnt[x-1] and not cnt[x+1] else '-'}")
    print()

x=10: lần=1, có 9? False, có 11? False -> cô đơn
x=6: lần=1, có 5? True, có 7? False -> -
x=5: lần=1, có 4? False, có 6? True -> -
x=8: lần=1, có 7? False, có 9? False -> cô đơn

x=1: lần=1, có 0? False, có 2? False -> cô đơn
x=3: lần=2, có 2? False, có 4? False -> -
x=5: lần=1, có 4? False, có 6? False -> cô đơn
x=3: lần=2, có 2? False, có 4? False -> -



## 5. Cách 1 — Brute Force / baseline: với mỗi số, quét cả mảng

**Ý tưởng:** đếm số lần `x` xuất hiện và kiểm tra `x ± 1` bằng cách duyệt lại cả mảng.

In [8]:
def brute(nums):
    ans = []
    for x in nums:
        cnt_x = sum(1 for y in nums if y == x)
        has_nb = any(y == x - 1 or y == x + 1 for y in nums)
        if cnt_x == 1 and not has_nb:
            ans.append(x)
    return ans

for nums, exp in examples:
    got = brute(nums)
    print(got, "OK" if same(got, exp) else "SAI")

[10, 8] OK
[1, 5] OK


**Độ phức tạp:** O(n²) — `n = 10⁵` ⇒ 10¹⁰ phép ⇒ TLE.

**Nút thắt:** câu hỏi "giá trị `v` xuất hiện bao nhiêu lần?" bị trả lời lại bằng một lượt quét O(n) mỗi lần. Đếm **một lần** cho mọi giá trị ⇒ bảng đếm.

In [9]:
for n in [1000, 3000]:
    a = [random.randint(0, 10**6) for _ in range(n)]
    t = time.perf_counter(); brute(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=1000: 84 ms


n=3000: 716 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** dựng bảng đếm `cnt` một lần (O(n)). Khi đó với mỗi giá trị **phân biệt** `x`:

`x` cô đơn ⇔ `cnt[x] == 1` **và** `cnt[x − 1] == 0` **và** `cnt[x + 1] == 0`.

Quy tắc:
- **Nếu** `cnt[x] ≥ 2` **thì** loại ngay (lặp lại).
- **Nếu** có hàng xóm `x ± 1` (dù hàng xóm đó lặp lại hay không) **thì** loại.
- Duyệt theo **khóa** của `cnt` (mỗi giá trị một lần) — vì `x` cô đơn thì nó chỉ xuất hiện một lần, duyệt theo khóa hay theo `nums` đều cho cùng tập đáp án, nhưng duyệt khóa tránh lặp.

**Cách sắp xếp:** sau khi sắp xếp `a`, `a[i]` cô đơn ⇔ `a[i−1] < a[i] − 1` (hoặc không có) **và** `a[i+1] > a[i] + 1` (hoặc không có). Một điều kiện "hiệu > 1 với cả hai hàng xóm" gói gọn cả "không lặp" (hiệu 0) lẫn "không kề" (hiệu 1).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + nhìn hàng xóm

In [10]:
def sort_neighbors(nums):
    a = sorted(nums); n = len(a); ans = []
    for i, x in enumerate(a):
        left_ok = i == 0 or a[i - 1] < x - 1
        right_ok = i == n - 1 or a[i + 1] > x + 1
        if left_ok and right_ok:
            ans.append(x)
    return ans

for nums, exp in examples:
    got = sort_neighbors(nums)
    print(got, "OK" if same(got, exp) else "SAI")

[8, 10] OK
[1, 5] OK


**Độ phức tạp cách 2:** O(n log n) thời gian, O(n) bộ nhớ (bản sắp xếp). Ưu điểm: kết quả **đã sắp xếp**, không cần bảng băm.

### 7.2 Cách 3 — `Counter` (cách doocs)

In [11]:
def counter_way(nums):
    cnt = Counter(nums)
    return [x for x, v in cnt.items() if v == 1 and cnt[x - 1] == 0 and cnt[x + 1] == 0]

for nums, exp in examples:
    got = counter_way(nums)
    print(got, "OK" if same(got, exp) else "SAI")

[10, 8] OK
[1, 5] OK


**Độ phức tạp cách 3:** O(n) thời gian trung bình, O(số giá trị phân biệt) bộ nhớ.

### 7.3 Cách 4 — Mảng đếm theo giá trị (không băm)

In [12]:
def count_array(nums):
    V = max(nums) + 2                       # đệm 1 ô cho x + 1
    cnt = [0] * V
    for x in nums:
        cnt[x] += 1
    ans = []
    for x in nums:
        if cnt[x] == 1 and cnt[x + 1] == 0 and (x == 0 or cnt[x - 1] == 0):
            ans.append(x)
    return ans

for nums, exp in examples:
    got = count_array(nums)
    print(got, "OK" if same(got, exp) else "SAI")

[10, 8] OK
[1, 5] OK


Dùng `max(nums) + 2` thay vì cố định `10⁶ + 2` để tiết kiệm bộ nhớ khi giá trị nhỏ. Điều kiện `x == 0 or …` tránh bẫy `cnt[-1]`.

**Độ phức tạp cách 4:** O(n + V) thời gian và O(V) bộ nhớ (`V ≤ 10⁶ + 2`).

### 7.4 Cách 5 — `set` cho "có tồn tại" + `Counter` cho "lặp lại"

Tách hai câu hỏi: tồn tại hàng xóm → `set`; xuất hiện đúng một lần → `Counter`. Cùng độ phức tạp với cách 3, chỉ khác cách tổ chức (rõ ràng hơn khi đọc).

In [13]:
def set_and_counter(nums):
    seen = set(nums); cnt = Counter(nums)
    return [x for x in seen if cnt[x] == 1 and (x - 1) not in seen and (x + 1) not in seen]

for nums, exp in examples:
    got = set_and_counter(nums)
    print(got, "OK" if same(got, exp) else "SAI")

[8, 10] OK
[1, 5] OK


### 7.5 Cách 6 — Sắp xếp + `groupby` theo khối

Sau khi sắp xếp, mỗi giá trị phân biệt là một khối `(giá trị, số lần)`. Giá trị `v` cô đơn ⇔ khối có độ dài 1, khối trước có giá trị `< v − 1`, khối sau có giá trị `> v + 1`. Cách này tách bạch "lặp" và "kề" rõ ràng hơn cách 2.

In [14]:
def sort_groupby(nums):
    blocks = [(k, len(list(g))) for k, g in groupby(sorted(nums))]
    ans = []
    for i, (v, c) in enumerate(blocks):
        if c != 1: continue
        if i > 0 and blocks[i - 1][0] == v - 1: continue
        if i + 1 < len(blocks) and blocks[i + 1][0] == v + 1: continue
        ans.append(v)
    return ans

for nums, exp in examples:
    print(sort_groupby(nums), "OK" if same(sort_groupby(nums), exp) else "SAI")

[8, 10] OK
[1, 5] OK


Ở đây so sánh bằng `== v − 1` (không phải `< v − 1`) vì các khối đã **phân biệt**: khối trước chỉ có thể `≤ v − 1`; nếu nó bằng đúng `v − 1` thì có hàng xóm.

## 8. TRACE từng bước

### 8.1 Trace cách 2 (sắp xếp) trên `[1, 3, 5, 3, 7, 8, 10]`

`a = [1, 3, 3, 5, 7, 8, 10]`

| i | a[i] | trái | phải | trái OK (a[i−1] < x−1)? | phải OK (a[i+1] > x+1)? | cô đơn |
|---|---|---|---|---|---|---|
| 0 | 1 | — | 3 | ✔ | 3 > 2 ✔ | ✔ |
| 1 | 3 | 1 | 3 | 1 < 2 ✔ | 3 > 4 ✘ | ✘ |
| 2 | 3 | 3 | 5 | 3 < 2 ✘ | | ✘ |
| 3 | 5 | 3 | 7 | ✔ | 7 > 6 ✔ | ✔ |
| 4 | 7 | 5 | 8 | ✔ | 8 > 8 ✘ | ✘ |
| 5 | 8 | 7 | 10 | 7 < 7 ✘ | | ✘ |
| 6 | 10 | 8 | — | 8 < 9 ✔ | ✔ | ✔ |

In [15]:
def trace_sort(nums):
    a = sorted(nums); n = len(a); ans = []
    print("a =", a)
    for i, x in enumerate(a):
        L = a[i - 1] if i > 0 else None; R = a[i + 1] if i + 1 < n else None
        lok = L is None or L < x - 1; rok = R is None or R > x + 1
        print(f"i={i}: x={x:>2}, trái={L}, phải={R} -> trái OK={lok}, phải OK={rok}{'  => CÔ ĐƠN' if lok and rok else ''}")
        if lok and rok: ans.append(x)
    return ans

print("Kết quả:", trace_sort([1, 3, 5, 3, 7, 8, 10]))

a = [1, 3, 3, 5, 7, 8, 10]
i=0: x= 1, trái=None, phải=3 -> trái OK=True, phải OK=True  => CÔ ĐƠN
i=1: x= 3, trái=1, phải=3 -> trái OK=True, phải OK=False
i=2: x= 3, trái=3, phải=5 -> trái OK=False, phải OK=True
i=3: x= 5, trái=3, phải=7 -> trái OK=True, phải OK=True  => CÔ ĐƠN
i=4: x= 7, trái=5, phải=8 -> trái OK=True, phải OK=False
i=5: x= 8, trái=7, phải=10 -> trái OK=False, phải OK=True
i=6: x=10, trái=8, phải=None -> trái OK=True, phải OK=True  => CÔ ĐƠN
Kết quả: [1, 5, 10]


### 8.2 Trace cách 3 (Counter)

In [16]:
def trace_counter(nums):
    cnt = Counter(nums); ans = []
    print("cnt =", dict(cnt))
    for x, v in cnt.items():
        reason = []
        if v != 1: reason.append(f"lặp {v} lần")
        if cnt[x - 1]: reason.append(f"có {x - 1}")
        if cnt[x + 1]: reason.append(f"có {x + 1}")
        print(f"x={x}: {'CÔ ĐƠN' if not reason else 'loại vì ' + ', '.join(reason)}")
        if not reason: ans.append(x)
    return ans

print("Kết quả:", trace_counter([1, 3, 5, 3, 7, 8, 10]))

cnt = {1: 1, 3: 2, 5: 1, 7: 1, 8: 1, 10: 1}
x=1: CÔ ĐƠN
x=3: loại vì lặp 2 lần
x=5: CÔ ĐƠN
x=7: loại vì có 8
x=8: loại vì có 7
x=10: CÔ ĐƠN
Kết quả: [1, 5, 10]


## 9. Code nộp LeetCode

In [17]:
from typing import List
from collections import Counter

class Solution:
    def findLonely(self, nums: List[int]) -> List[int]:
        cnt = Counter(nums)
        ans = []
        for x, v in cnt.items():
            if v == 1 and cnt[x - 1] == 0 and cnt[x + 1] == 0:
                ans.append(x)
        return ans

In [18]:
sol = Solution()
for nums, exp in examples:
    got = sol.findLonely(nums)
    print(got, "| kỳ vọng (tùy thứ tự)", exp, "| OK" if same(got, exp) else "| SAI")

[10, 8] | kỳ vọng (tùy thứ tự) [10, 8] | OK
[1, 5] | kỳ vọng (tùy thứ tự) [1, 5] | OK


## 10. Vì sao thuật toán đúng

**Cách 3.** `Counter` cho `cnt[v]` = số lần xuất hiện chính xác của mọi `v` (0 nếu không xuất hiện). Định nghĩa "cô đơn" chính là `cnt[x] = 1 ∧ cnt[x−1] = 0 ∧ cnt[x+1] = 0`. Duyệt qua mọi khóa `x` (mọi giá trị xuất hiện) ⇒ thu được **đúng** tập số cô đơn; mỗi số cô đơn chỉ có một bản nên không bị trùng trong đáp án. ∎

**Cách 2.** Trong mảng đã sắp xếp `a`, xét `x = a[i]`.
- Nếu `x` lặp lại thì một bản khác nằm ở `a[i−1]` hoặc `a[i+1]` với hiệu 0 ⇒ điều kiện "hiệu > 1" sai ✔.
- Nếu `x − 1` tồn tại thì, do sắp xếp, phần tử lớn nhất nhỏ hơn `x` là `x − 1` và nó nằm ngay trước khối các bản `x` ⇒ (khi `x` không lặp) chính là `a[i−1]` ⇒ `a[i−1] = x − 1` không `< x − 1` ✔. Đối xứng cho `x + 1`.
- Ngược lại, nếu `x` cô đơn thì `a[i−1] ≤ x − 2` và `a[i+1] ≥ x + 2` ⇒ thỏa ✔. ∎

## 11. Phân tích Time / Space Complexity

`n = len(nums)`, `V = max(nums) ≤ 10⁶`, `D` = số giá trị phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | O(n²) | O(1) |
| 2. Sắp xếp + hàng xóm | O(n log n) | O(n) |
| 3. Counter | O(n) | O(D) |
| 4. Mảng đếm | O(n + V) | O(V) |
| 5. set + Counter | O(n) | O(D) |
| 6. Sắp xếp + groupby | O(n log n) | O(n) |

In [19]:
rng = random.Random(2150)
big = [rng.randint(0, 10**6) for _ in range(10**5)]
res = {}
for name, f in [("sort_neighbors", sort_neighbors), ("counter_way", counter_way), ("count_array", count_array),
                ("set_and_counter", set_and_counter), ("Solution", Solution().findLonely)]:
    t = time.perf_counter(); res[name] = sorted(f(big)); dt = time.perf_counter() - t
    print(f"{name:<16} -> {len(res[name])} số cô đơn, {dt*1000:.0f} ms")
assert len({tuple(v) for v in res.values()}) == 1

sort_neighbors   -> 74301 số cô đơn, 33 ms
counter_way      -> 74301 số cô đơn, 50 ms


count_array      -> 74301 số cô đơn, 53 ms


set_and_counter  -> 74301 số cô đơn, 51 ms


Solution         -> 74301 số cô đơn, 44 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "sort_neighbors": sort_neighbors, "sort_groupby": sort_groupby, "counter_way": counter_way,
       "count_array": count_array, "set_and_counter": set_and_counter, "Solution": Solution().findLonely}
edge = examples + [
    ([0], [0]),
    ([0, 1], []),
    ([0, 2], [0, 2]),
    ([5, 5], []),
    ([1000000], [1000000]),
    ([999999, 1000000], []),
    ([4, 4, 6], [6]),               # 4 lặp: không cô đơn; 6 không có 5, 7
    ([4, 4, 5], []),                # 5 có hàng xóm 4 (dù 4 lặp)
    ([0, 0, 2], [2]),
    ([1, 3, 5, 7, 9], [1, 3, 5, 7, 9]),
]
for nums, exp in edge:
    for name, f in ALL.items():
        a = list(nums)
        assert same(f(a), exp), (name, nums, f(list(nums)), exp)
        assert a == nums
print("Edge case OK:", len(edge))

Edge case OK: 12


### 12.2 Đối chiếu ngẫu nhiên với brute force (giá trị nhỏ để hay có trùng/kề)

In [21]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 15); hi = rng.choice([3, 10, 30, 10**6])
    nums = [rng.randint(0, hi) for _ in range(n)]
    ref = sorted(brute(nums))
    for name, f in ALL.items():
        got = f(list(nums))
        assert sorted(got) == ref, (name, nums)
        assert len(got) == len(set(got)), (name, "trùng trong đáp án")
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Tính chất: mọi số cô đơn xuất hiện đúng 1 lần; thêm `x+1` cho một số cô đơn `x` thì `x` hết cô đơn

In [22]:
rng = random.Random(7)
for _ in range(500):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 30))]
    lonely = Solution().findLonely(nums)
    assert all(nums.count(x) == 1 for x in lonely)
    if lonely:
        x = rng.choice(lonely)
        assert x not in Solution().findLonely(nums + [x + 1])
print("tính chất OK")

tính chất OK


### 12.4 Cực trị theo ràng buộc

- `n = 10⁵` số đều bằng nhau ⇒ không có số cô đơn.
- `n = 10⁵` số chẵn phân biệt `0, 2, 4, …` ⇒ **mọi** số đều cô đơn.
- Giá trị lớn nhất `10⁶` đứng một mình ⇒ cô đơn (kiểm tra `x + 1 = 10⁶ + 1` không lỗi chỉ số trong mảng đếm).

In [23]:
same_vals = [7] * 10**5
evens = list(range(0, 2 * 10**5, 2))
for name, f in ALL.items():
    if name == "brute": continue
    assert f(same_vals) == []
    assert sorted(f(evens)) == evens
    assert same(f([10**6, 3, 3]), [10**6])
print("cực trị OK")

cực trị OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện "xuất hiện đúng một lần"

In [24]:
def wrong_no_count(nums):
    s = set(nums)
    return [x for x in s if x - 1 not in s and x + 1 not in s]        # SAI: không kiểm tra lặp

print("bản sai :", sorted(wrong_no_count([1, 3, 5, 3])))
print("bản đúng:", sorted(Solution().findLonely([1, 3, 5, 3])))

bản sai : [1, 3, 5]
bản đúng: [1, 5]


**Sửa:** thêm `cnt[x] == 1`.

### Lỗi 2 — Duyệt theo `nums` khi điều kiện bị nới (trả trùng)

Nếu ai đó "sửa" Lỗi 1 bằng cách duyệt `nums` thay cho `set` mà vẫn quên điều kiện đếm, kết quả còn chứa **bản trùng**:

In [25]:
def wrong_dup_output(nums):
    s = set(nums)
    return [x for x in nums if x - 1 not in s and x + 1 not in s]

print("bản sai :", wrong_dup_output([1, 3, 5, 3]))
print("bản đúng:", Solution().findLonely([1, 3, 5, 3]))

bản sai : [1, 3, 5, 3]
bản đúng: [1, 5]


**Sửa:** điều kiện đúng-một-lần tự động loại trùng; duyệt theo khóa của `Counter` càng chắc chắn.

### Lỗi 3 — Mảng đếm: dùng `cnt[x - 1]` khi `x = 0` (chỉ số −1 trỏ về cuối mảng)

In [26]:
def wrong_negative_index(nums):
    V = max(nums) + 1                                # bảng vừa khít: chỉ số 0..max
    cnt = [0] * V
    for x in nums: cnt[x] += 1
    return [x for x in nums
            if cnt[x] == 1 and (x + 1 >= V or cnt[x + 1] == 0) and cnt[x - 1] == 0]   # cnt[-1] khi x = 0!

print("[0, 5]    - bản sai:", wrong_negative_index([0, 5]), "| đúng:", Solution().findLonely([0, 5]))
print("[0, 3, 7] - bản sai:", wrong_negative_index([0, 3, 7]), "| đúng:", Solution().findLonely([0, 3, 7]))

[0, 5]    - bản sai: [5] | đúng: [0, 5]
[0, 3, 7] - bản sai: [3, 7] | đúng: [0, 3, 7]


**Vì sao sai:** bảng có kích thước `max + 1`, nên `cnt[0 − 1] = cnt[−1] = cnt[max]` — chính là số lần xuất hiện của **phần tử lớn nhất** (luôn ≥ 1)! Vì vậy số 0 không bao giờ được coi là cô đơn. Python không báo lỗi với chỉ số âm, nên lỗi "im lặng". (Bản `max + 2` ở cách 4 có ô cuối luôn bằng 0 nên may mắn tránh được — nhưng đừng dựa vào may mắn.) **Sửa:** `x == 0 or cnt[x − 1] == 0`, hoặc dịch mọi giá trị +1 khi lập bảng.

### Lỗi 4 — Sắp xếp nhưng chỉ so một phía

In [27]:
def wrong_one_side(nums):
    a = sorted(nums); ans = []
    for i, x in enumerate(a):
        if i + 1 == len(a) or a[i + 1] > x + 1:           # SAI: quên phía trái
            ans.append(x)
    return ans

print("bản sai :", wrong_one_side([10, 6, 5, 8]))
print("bản đúng:", sorted(Solution().findLonely([10, 6, 5, 8])))

bản sai : [6, 8, 10]
bản đúng: [8, 10]


**Sửa:** kiểm tra cả `a[i−1] < x − 1` (hoặc `i == 0`) **và** `a[i+1] > x + 1` (hoặc `i == n−1`).

### Lỗi 5 — Dùng `defaultdict(int)` và truy cập `cnt[x ± 1]` trong lúc duyệt `cnt.items()`

In [28]:
from collections import defaultdict
def wrong_defaultdict(nums):
    cnt = defaultdict(int)
    for x in nums: cnt[x] += 1
    try:
        return [x for x, v in cnt.items() if v == 1 and cnt[x - 1] == 0 and cnt[x + 1] == 0]
    except RuntimeError as e:
        return f"RuntimeError: {e}"

print("bản sai :", wrong_defaultdict([10, 6, 5, 8]))
print("bản đúng:", Solution().findLonely([10, 6, 5, 8]))

bản sai : RuntimeError: dictionary changed size during iteration
bản đúng: [10, 8]


**Sửa:** dùng `Counter` (truy cập khóa thiếu không chèn), hoặc `cnt.get(x − 1, 0)`, hoặc duyệt `list(cnt.items())`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Định nghĩa dựa trên "xuất hiện bao nhiêu lần" + "giá trị lân cận có mặt không" ⇒ **bảng đếm** (`Counter`).
- Giá trị nhỏ (≤ 10⁶) ⇒ có thể dùng **mảng đếm**; cẩn thận chỉ số âm trong Python.
- Sắp xếp làm các phần tử "liên quan" nằm cạnh nhau ⇒ chỉ cần nhìn **hàng xóm**.
- Đáp án "thứ tự tùy ý" ⇒ test bằng `sorted(...)`.
- Bài cùng họ: 128 Longest Consecutive Sequence (set + kiểm tra `x − 1`), 1748 Sum of Unique Elements, 387 First Unique Character, 2341.

## 15. Biến thể của bài

### 15.1 Biến thể A — "Cô đơn bán kính `r`": không có số nào trong `[x − r, x + r]` ngoài chính nó

Sắp xếp rồi so hàng xóm với khoảng cách `> r`. (Với `r = 1` là bài gốc.)

In [29]:
def lonely_radius(nums, r):
    a = sorted(nums); n = len(a)
    return [x for i, x in enumerate(a)
            if (i == 0 or a[i - 1] < x - r) and (i == n - 1 or a[i + 1] > x + r)]

def lonely_radius_brute(nums, r):
    return sorted(x for i, x in enumerate(nums)
                  if all(not (x - r <= y <= x + r) for j, y in enumerate(nums) if j != i))

print(lonely_radius([1, 4, 8, 9, 15], 2), lonely_radius([1, 4, 8, 9, 15], 1))
rng = random.Random(151)
for _ in range(500):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 12))]; r = rng.randint(0, 4)
    assert lonely_radius(nums, r) == lonely_radius_brute(nums, r)
print("khớp brute force")

[1, 4, 15] [1, 4, 15]
khớp brute force


Lưu ý `r = 0`: "cô đơn" chỉ còn nghĩa là **không lặp** — đúng là LC 1748 (phần tử duy nhất).

### 15.2 Biến thể B — Đếm số cô đơn trong **mọi cửa sổ trượt** dài `w`

Duy trì `Counter` của cửa sổ; khi thêm/bớt một giá trị `v`, chỉ trạng thái cô đơn của `v − 1, v, v + 1` có thể đổi ⇒ cập nhật O(1) mỗi bước.

In [30]:
def lonely_in_windows(nums, w):
    cnt = Counter(); lonely = 0; out = []
    def is_l(x): return cnt[x] == 1 and cnt[x - 1] == 0 and cnt[x + 1] == 0
    def change(v, delta):
        nonlocal lonely
        affected = (v - 1, v, v + 1)
        lonely -= sum(is_l(x) for x in affected)
        cnt[v] += delta
        lonely += sum(is_l(x) for x in affected)
    for i, v in enumerate(nums):
        change(v, +1)
        if i >= w: change(nums[i - w], -1)
        if i >= w - 1: out.append(lonely)
    return out

def lonely_in_windows_brute(nums, w):
    return [len(Solution().findLonely(nums[i:i + w])) for i in range(len(nums) - w + 1)]

print(lonely_in_windows([1, 3, 5, 3, 7, 8, 10], 3))
rng = random.Random(152)
for _ in range(400):
    nums = [rng.randint(0, 12) for _ in range(rng.randint(1, 20))]; w = rng.randint(1, len(nums))
    assert lonely_in_windows(nums, w) == lonely_in_windows_brute(nums, w)
print("khớp brute force")

[3, 1, 3, 1, 1]
khớp brute force


### 15.3 Biến thể C — Trả về theo **thứ tự xuất hiện** trong `nums` (thay vì tùy ý)

Duyệt `nums` (không phải khóa của `Counter`) — vì số cô đơn chỉ xuất hiện một lần nên không lo trùng.

In [31]:
def lonely_in_order(nums):
    cnt = Counter(nums)
    return [x for x in nums if cnt[x] == 1 and not cnt[x - 1] and not cnt[x + 1]]
print(lonely_in_order([10, 6, 5, 8]), lonely_in_order([9, 1, 30, 3, 3]))

[10, 8] [9, 1, 30]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | quét lại cả mảng | O(n²) | O(1) | kiểm chứng |
| 2. Sắp xếp + hàng xóm | hiệu > 1 hai phía | O(n log n) | O(n) | cần đáp án sắp xếp, biến thể bán kính |
| 3. Counter | ba điều kiện bằng bảng đếm | O(n) | O(D) | **mặc định** |
| 4. Mảng đếm | không băm | O(n + V) | O(V) | giá trị nhỏ, ngôn ngữ khác |
| 5. set + Counter | tách "tồn tại" và "lặp" | O(n) | O(D) | dễ đọc |
| 6. Sắp xếp + groupby | khối giá trị phân biệt | O(n log n) | O(n) | thích tư duy theo khối |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `findLonely([0, 0, 1, 3])` và `findLonely([2, 4, 6, 6, 8])`?

**Bài 2 (tính tay):** Với `[7, 1, 9, 1, 4, 11, 12]`, liệt kê số cô đơn và lý do loại từng số còn lại.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về **số lượng** số cô đơn bằng cách sắp xếp (không dùng `Counter`/`set`).

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` dùng mảng đếm theo giá trị, xử lý đúng biên `x = 0`.

**Bài 5 (mini challenge):** Được **xóa đúng một phần tử** khỏi `nums`. Viết `bt5_todo(nums)` trả về **số lượng số cô đơn lớn nhất** có thể đạt sau khi xóa (`n ≥ 2`).

In [32]:
def bt3_todo(nums):
    # TODO: đếm số cô đơn bằng sắp xếp
    pass

def bt4_todo(nums):
    # TODO: mảng đếm theo giá trị
    pass

def bt5_todo(nums):
    # TODO: xóa một phần tử để số cô đơn lớn nhất
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đếm. Mức 2: `0` lặp; `1` có hàng xóm 0 (dù 0 lặp); `3` không có 2, 4. Mức 3: `[3]`; ví dụ hai: `2` (không có 1, 3) ✔, `4` ✔, `6` lặp ✘, `8` ✔ ⇒ `[2, 4, 8]`.

**Bài 2.** Mức 1: 1 lặp. Mức 2: 11 và 12 kề nhau. Mức 3: cô đơn: `7, 9, 4`.

**Bài 3.** Mức 1: như cách 2. Mức 2: thay `append` bằng đếm. Mức 3: `sum(1 for i ... if left_ok and right_ok)`.

**Bài 4.** Mức 1: `cnt = [0] * (max + 2)`. Mức 2: điều kiện `x == 0 or cnt[x-1] == 0`. Mức 3: duyệt `nums`, thêm `x` khi đủ ba điều kiện.

**Bài 5.** Mức 1: xóa một phần tử chỉ ảnh hưởng trạng thái của `v − 1, v, v + 1`. Mức 2: thử xóa từng **giá trị phân biệt** `v` (xóa bản nào của `v` cũng như nhau), tính delta trên ba giá trị bị ảnh hưởng. Mức 3: đáp án = base + max delta (O(D)); kiểm chứng bằng brute force thử xóa từng chỉ số.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[3]`, `[2, 4, 8]`. **Bài 2:** cô đơn `7, 9, 4`; `1` lặp, `11` có 12, `12` có 11.

In [33]:
assert same(Solution().findLonely([0, 0, 1, 3]), [3])
assert same(Solution().findLonely([2, 4, 6, 6, 8]), [2, 4, 8])
assert same(Solution().findLonely([7, 1, 9, 1, 4, 11, 12]), [7, 9, 4])

def bt3(nums):
    a = sorted(nums); n = len(a)
    return sum(1 for i, x in enumerate(a) if (i == 0 or a[i - 1] < x - 1) and (i == n - 1 or a[i + 1] > x + 1))

def bt4(nums):
    cnt = [0] * (max(nums) + 2)
    for x in nums: cnt[x] += 1
    return [x for x in nums if cnt[x] == 1 and cnt[x + 1] == 0 and (x == 0 or cnt[x - 1] == 0)]

def bt5(nums):
    cnt = Counter(nums)
    def is_l(x): return cnt[x] == 1 and cnt[x - 1] == 0 and cnt[x + 1] == 0
    base = sum(is_l(x) for x in list(cnt))
    best = None
    for v in list(cnt):
        aff = (v - 1, v, v + 1)
        before = sum(is_l(x) for x in aff)
        cnt[v] -= 1
        after = sum(is_l(x) for x in aff)
        cnt[v] += 1
        val = base - before + after
        best = val if best is None else max(best, val)
    return best

def bt5_brute(nums):
    return max(len(Solution().findLonely(nums[:i] + nums[i + 1:])) for i in range(len(nums)))

rng = random.Random(1950)
for _ in range(2000):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(2, 12))]
    ref = sorted(brute(nums))
    assert bt3(nums) == len(ref) and sorted(bt4(nums)) == ref
    assert bt5(nums) == bt5_brute(nums), nums
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: `cnt[v] -= 1` có thể làm `cnt[v] = 0` — khi đó `v` "biến mất", và `v − 1`, `v + 1` có thể **trở thành** cô đơn. `Counter` vẫn giữ khóa `v` với giá trị 0 nhưng `is_l` đọc đúng nghĩa (0 lần = không có). Cuối mỗi lần thử ta trả lại `cnt[v]`.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = Counter(nums)
return [x for x, v in cnt.items()
        if v == 1 and cnt[x - 1] == 0 and cnt[x + 1] == 0]
```

**Checklist:**
- [ ] Đủ **ba** điều kiện: đúng một lần, không có `x − 1`, không có `x + 1`.
- [ ] Số lặp lại vẫn làm hàng xóm của nó mất cô đơn.
- [ ] Dùng `Counter`/`.get` (không `defaultdict` khi duyệt).
- [ ] Mảng đếm: xử lý `x = 0` (không dùng `cnt[-1]`).
- [ ] Đáp án thứ tự tùy ý — test bằng `sorted`.